# Per-recording Evaluation
Notebook version of `test_per_recording.py`.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score

# Configure inputs
base_dir = "output"
methods = ["file_name"]  # CSV stems without .csv


In [2]:
output = []

for method in methods:
    df = pd.read_csv(os.path.join(base_dir, f"{method}.csv"), header=0)
    df["y_pred"] = df["y_score"] > 0.5
    per_record = df.groupby(by="subject").apply(lambda d: d["y_pred"].mean() * 60)
    per_record.name = method
    output.append(per_record)

output = pd.concat(output, axis=1)
output.head()


/tmp/ipykernel_1572428/4134521829.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_record = df.groupby(by="subject").apply(lambda d: d["y_pred"].mean() * 60)


,file_name
subject,
a01,58.723404
a01r,55.764706
a02,52.173913
a02r,40.714286
a03,26.526316


In [3]:
from pathlib import Path

# Resolve metadata file robustly across common layouts
candidates = [
    Path("dataset/additional-information.txt"),
    Path("dataset/1.0.0/additional-information.txt"),
]
meta_path = next((c for c in candidates if c.exists()), None)
if meta_path is None:
    raise FileNotFoundError(
        "Could not find additional-information.txt. Tried: "
        + ", ".join(str(c) for c in candidates)
    )

with open(meta_path, "r") as f:
    original = []
    for line in f:
        rows = line.strip().split("	")
        if len(rows) == 12 and rows[0].startswith("x"):
            original.append([rows[0], float(rows[3]) / float(rows[1]) * 60])

original = pd.DataFrame(original, columns=["subject", "original"]).set_index("subject")
all_df = pd.concat((output, original), axis=1)

corr = all_df.corr()
all_bin = all_df.applymap(lambda a: int(a > 5))

result = []
for method in methods:
    C = confusion_matrix(all_bin["original"], all_bin[method], labels=(1, 0))
    TP, TN, FP, FN = C[0, 0], C[1, 1], C[1, 0], C[0, 1]
    acc = 1.0 * (TP + TN) / (TP + TN + FP + FN)
    sn = 1.0 * TP / (TP + FN)
    sp = 1.0 * TN / (TN + FP)
    auc = roc_auc_score(all_df["original"] > 5, all_df[method])
    result.append([method, acc * 100, sn * 100, sp * 100, auc, corr["original"][method]])

np.savetxt(
    os.path.join(base_dir, "Table 3.csv"),
    result,
    fmt="%s",
    delimiter=",",
    comments="",
    header="Method,Accuracy(%),Sensitivity(%),Specificity(%),AUC,Corr",
)

result_df = pd.DataFrame(
    result,
    columns=["Method", "Accuracy(%)", "Sensitivity(%)", "Specificity(%)", "AUC", "Corr"],
)
print(f"Using metadata: {meta_path}")
print(f"Saved: {os.path.join(base_dir, 'Table 3.csv')}")
result_df


Using metadata: dataset/1.0.0/additional-information.txt
Saved: output/Table 3.csv


/tmp/ipykernel_1572428/1883656357.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  all_bin = all_df.applymap(lambda a: int(a > 5))


,Method,Accuracy(%),Sensitivity(%),Specificity(%),AUC,Corr
0,file_name,64.102564,100.0,49.090909,0.734387,0.993232


In [4]:
# Threshold sweep: tune segment and record thresholds on current CSV
# Note: tuning/evaluating on the same data can overestimate performance.

seg_thresholds = np.round(np.arange(0.30, 0.91, 0.02), 2)
record_thresholds = np.arange(1, 31)  # AHI cutoff candidates

# Resolve metadata file robustly
from pathlib import Path
candidates = [
    Path("dataset/additional-information.txt"),
    Path("dataset/1.0.0/additional-information.txt"),
]
meta_path = next((c for c in candidates if c.exists()), None)
if meta_path is None:
    raise FileNotFoundError(
        "Could not find additional-information.txt. Tried: "
        + ", ".join(str(c) for c in candidates)
    )

with open(meta_path, "r") as f:
    original = []
    for line in f:
        rows = line.strip().split("	")
        if len(rows) == 12 and rows[0].startswith("x"):
            original.append([rows[0], float(rows[3]) / float(rows[1]) * 60])
original = pd.DataFrame(original, columns=["subject", "original"]).set_index("subject")

def evaluate_thresholds(df, seg_thr, rec_thr):
    tmp = df.copy()
    tmp["y_pred"] = tmp["y_score"] > seg_thr
    per_record = tmp.groupby("subject").apply(lambda d: d["y_pred"].mean() * 60)
    all_df = pd.concat([per_record.rename("pred"), original], axis=1).dropna()

    y_true = (all_df["original"] > rec_thr).astype(int)
    y_hat = (all_df["pred"] > rec_thr).astype(int)

    C = confusion_matrix(y_true, y_hat, labels=(1, 0))
    TP, TN, FP, FN = C[0, 0], C[1, 1], C[1, 0], C[0, 1]
    acc = (TP + TN) / (TP + TN + FP + FN + 1e-8)
    sn = TP / (TP + FN + 1e-8)
    sp = TN / (TN + FP + 1e-8)
    auc = roc_auc_score(y_true, all_df["pred"]) if len(np.unique(y_true)) > 1 else np.nan
    corr = all_df.corr().loc["original", "pred"]
    bal_acc = 0.5 * (sn + sp)
    return acc, sn, sp, auc, corr, bal_acc, C

rows = []
for method in methods:
    df = pd.read_csv(os.path.join(base_dir, f"{method}.csv"), header=0)

    best = None
    for seg_thr in seg_thresholds:
        for rec_thr in record_thresholds:
            acc, sn, sp, auc, corr, bal_acc, C = evaluate_thresholds(df, seg_thr, rec_thr)
            score = (bal_acc, acc)  # prioritize balanced accuracy, then accuracy
            if (best is None) or (score > best["score"]):
                best = {
                    "method": method,
                    "seg_thr": float(seg_thr),
                    "rec_thr": int(rec_thr),
                    "acc": acc,
                    "sn": sn,
                    "sp": sp,
                    "auc": auc,
                    "corr": corr,
                    "bal_acc": bal_acc,
                    "C": C,
                    "score": score,
                }

    rows.append([
        best["method"], best["seg_thr"], best["rec_thr"],
        best["acc"] * 100, best["sn"] * 100, best["sp"] * 100,
        best["auc"], best["corr"], best["bal_acc"] * 100
    ])

result_df = pd.DataFrame(
    rows,
    columns=[
        "Method", "SegmentThreshold", "RecordThreshold(AHI)",
        "Accuracy(%)", "Sensitivity(%)", "Specificity(%)",
        "AUC", "Corr", "BalancedAccuracy(%)"
    ],
)

# Save tuned results
out_main = os.path.join(base_dir, "Table 3.csv")
out_tuned = os.path.join(base_dir, "Table 3 tuned.csv")
result_df.to_csv(out_main, index=False)
result_df.to_csv(out_tuned, index=False)

print(f"Using metadata: {meta_path}")
print(f"Saved: {out_main}")
print(f"Saved: {out_tuned}")
result_df


/tmp/ipykernel_1572428/4035338996.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_record = tmp.groupby("subject").apply(lambda d: d["y_pred"].mean() * 60)
/tmp/ipykernel_1572428/4035338996.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_record = tmp.groupby("subject").apply(lambda d: d["y_pred"].mean() * 60)
/tmp/ipykernel_1572428/4035338996.py:31: FutureWarning: DataFrameGroupBy.apply operated on t

Using metadata: dataset/1.0.0/additional-information.txt
Saved: output/Table 3.csv
Saved: output/Table 3 tuned.csv


/tmp/ipykernel_1572428/4035338996.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_record = tmp.groupby("subject").apply(lambda d: d["y_pred"].mean() * 60)
/tmp/ipykernel_1572428/4035338996.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_record = tmp.groupby("subject").apply(lambda d: d["y_pred"].mean() * 60)
/tmp/ipykernel_1572428/4035338996.py:31: FutureWarning: DataFrameGroupBy.apply operated on t

,Method,SegmentThreshold,RecordThreshold(AHI),Accuracy(%),Sensitivity(%),Specificity(%),AUC,Corr,BalancedAccuracy(%)
0,file_name,0.3,22,100.0,100.0,100.0,1.0,0.987396,100.0
